# Lab 4: Transparency with Chain-of-Thought (CoT) Prompting

**Duration:** ~15 minutes

---

Labs 1–3 explored transparency for ML models and RAG-grounded LLMs. This lab focuses on **transparency through prompting**.

**Chain-of-Thought (CoT)** prompting instructs a model to reason step by step before producing a final answer. This provides two XAI benefits:

1. **Correctness**: forcing step-by-step reasoning reduces errors and wrong assumptions, ultimately improving accuracy.
2. **Interpretability**: the reasoning trace shows *how* the model reached its conclusion, not just *what* it concluded.

In this lab you will:

1. Observe how LLMs fail on reasoning tasks without guidance (**baseline**)
2. Improve results with **Chain-of-Thought prompting**
3. Handle a subtler failure (*motivated reasoning*) using **prompt chaining**
4. Explore multiple reasoning paths with **Tree of Thoughts (ToT)**
5. Run prompts that expose reasoning failures even in modern frontier models (Gemini, Claude Sonnet)

## Prerequisites

- [Ollama](https://ollama.com) running locally with `qwen3:0.6b` pulled
- Python package: `requests`

## Step 1: Imports and Configuration

The setup mirrors Lab 3. `qwen3:0.6b` may produce `<think>...</think>` blocks when using its internal reasoning mode. For now, we strip them so the visible output only contains what the model explicitly states, making it easier to compare responses with and without CoT.

In [19]:
import re
from urllib import response
import requests

OLLAMA_URL = 'http://localhost:11434/api/chat'
OLLAMA_MODEL = 'qwen3:0.6b'


def call_llm(prompt, temperature=0.7):
    """Call the local LLM and return the response text."""
    response = requests.post(
        OLLAMA_URL,
        json={
            'model': OLLAMA_MODEL,
            'messages': [{'role': 'user', 'content': prompt}],
            'stream': False,
            'options': {'temperature': temperature},
        },
        timeout=60,
    )
    response.raise_for_status()
    raw = response.json()['message']['content']
    # Strip <think>...</think> reasoning blocks that qwen3 models produce
    # return re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    return response.json()['message']['content'].strip()


print(f'Model: {OLLAMA_MODEL}')
print('Configuration loaded.')

Model: qwen3:0.6b
Configuration loaded.


## Step 2: Reasoning Failures Without CoT

LLMs are trained to produce fluent, confident text. When asked a reasoning question directly, small models often rely on **pattern matching** (recognising what kind of answer usually follows this kind of question) rather than genuine reasoning.

Two examples below reveal this clearly on `qwen3:0.6b`.

### Example 1: Simple Math Problem

LLM optimize for the next token that is more likely to appear. This works well for text. But arithmetic is rule-based. Deterministic. There is no leeway in the token that should follow "12.24 x 33.02 =", a close enough answer will be wrong. Because of this and LLMs' intrinsic non-deterministic nature, they tend to struggle with simple arithmetic problems.

Consider the following simple math problem, that a primary school kid would be able to solve:

In [20]:
problem_math = (
    'The cafeteria had 264 apples. '
    'They used 82 to make lunch, '
    'students bought 43 of them, '
    '12 were rotten, '
    '14 were eaten by the staff, '
    'and they purchased 120 more.'
    'How many apples are left? '
    'Give only a number as answer.'
)

print('Question:', problem_math)
print('\nAnswer (no CoT):')
print(call_llm(problem_math))

Question: The cafeteria had 264 apples. They used 82 to make lunch, students bought 43 of them, 12 were rotten, 14 were eaten by the staff, and they purchased 120 more.How many apples are left? Give only a number as answer.

Answer (no CoT):
333


When in reality, the answer is:

$$264 - 82 - 43 - 12 -14 + 120 = 233 \text{ apples}$$

Some more modern models (also qwen) support calculations via separate circuits. However, these kinds of math problems come with a semantical challenge: "students bought" means subtracting, "they [cafeteria employees] purchased" means adding. But the models tend to interpret them equally, since they both talk about buying/purchasing. 

### Example 2: Character Counting

Counting individual characters forces a model to process a word symbol by symbol. This is something that tokenisation works against. Without explicit guidance, LLMs tend to rely on memorised approximate counts rather than actually counting.

*Correct answer: **3** — s·t·**r**·a·w·b·e·**r**·**r**·y*

In [5]:
problem_letters = (
    "How many times does the letter 'r' appear in the word 'strawberry'? "
    'Give only a number as your answer.'
)

print('Question:', problem_letters)
print('\nAnswer (no CoT):')
print(call_llm(problem_letters))

Question: How many times does the letter 'r' appear in the word 'strawberry'? Give only a number as your answer.

Answer (no CoT):
2


> **Observe:** the answers above are likely wrong (or correct by coincidence, but not by deterministic calculations). The model has no obligation to show its work, so any error in its implicit reasoning is invisible. Therefore, we cannot tell whether a correct answer was actually derived correctly.

## Step 3: Chain-of-Thought Prompting

Adding the instruction *"Think through this step by step"* to a prompt is often enough to trigger more deliberate reasoning. This is **zero-shot CoT**: no examples are provided, only the instruction.

CoT was first systematically studied by Wei et al. (2022). They showed that adding this simple phrase to prompts substantially improved performance on multi-step reasoning benchmarks, especially for larger models.

In [6]:
def add_cot(problem):
    """Append a chain-of-thought instruction to a prompt."""
    return problem + '\nThink through this step by step before giving your final answer.'


print('=== Example 1: Simple Math Problem (with CoT) ===')
print(call_llm(add_cot(problem_math)))

print()
print('=' * 55)
print('=== Example 2: Letter Counting (with CoT) ===')
print(call_llm(add_cot(problem_letters)))

=== Example 1: Simple Math Problem (with CoT) ===
To determine how many apples are left after all the operations, let's carefully go through each step:

1. **Initial apples**: 264 apples were present.
2. **Used for lunch**: 82 apples were used.
   - $264 - 82 = 182$ apples left after lunch.
3. **Students bought**: 43 apples were bought.
   - $182 + 43 = 225$ apples after students bought.
4. **Rotten apples**: 12 apples were rotten.
   - $225 - 12 = 213$ apples after accounting for rotten apples.
5. **Staff ate**: 14 apples were eaten by the staff.
   - $213 - 14 = 209$ apples after subtracting the staff's consumption.
6. **Purchased more**: 120 apples were purchased.
   - $209 + 120 = 329$ apples left.

Thus, the final number of apples left is:

$$
\boxed{329}
$$

=== Example 2: Letter Counting (with CoT) ===
To determine how many times the letter 'r' appears in the word "strawberry", we examine the word letter by letter:

- S, T, R, A, W, B, E, R, R, Y

Counting the occurrences of the

> **Observe:**
> - The reasoning trace makes the model's logic visible. Even when the final answer is still wrong, you can now see *where* the reasoning goes astray, which is itself a transparency gain.
> - For the letter-counting problem, CoT often helps because the model is guided to enumerate characters one by one rather than retrieve a memorised answer.
> - For the math problem, results may vary. Small models can still get it wrong even with CoT; larger models benefit more. Try running the cell a few times to see the variation.

For the math problem, zero-shot CoT might not be enough. What if we use few-shot and add a similar example? Does it arrive to the correct result? What if we run this query in another (frontier) model, like via ChatGPT, Gemini or Claude?

In [12]:
print(call_llm(problem_math + 
               "\nThink through this step by step before giving your final answer. See this similar example: "
               "\n'The cafeteria had 100 oranges. "
               "They used 20 to make juice, (they are deducted)"
               "students bought 30 of them from the cafeteria to eat them, (oranges bought by students are deducted) "
               "10 went bad, (they are deducted) and they bought 50 more for the next day (they are added). "
               "How many oranges are left?'"
               "\nAnswer: 100 oranges, plus 20 to make juice, minus 30 that students ate, minus 10 that went bad, plus 50 purchased = 90."))

The cafeteria started with 264 apples. They used 82 for lunch, then students bought 43 apples, 12 were rotten, 14 were eaten by staff, and they purchased 120 more. The steps are as follows:

1. Start with 264 apples.  
2. Subtract 82 used for lunch: $264 - 82 = 182$.  
3. Subtract 43 apples bought by students: $182 - 43 = 139$.  
4. Subtract 12 rotten apples: $139 - 12 = 127$.  
5. Subtract 14 apples eaten by staff: $127 - 14 = 113$.  
6. Add 120 new apples purchased: $113 + 120 = 233$.  

Final answer: 233.


## Step 4: Prompt Chaining (When CoT Fails)

CoT is powerful but not foolproof. A subtler failure mode is **motivated reasoning**: the model has a strong prior about the expected answer (from training data) and unconsciously steers its step-by-step reasoning toward that answer, regardless of if it's right or wrong.

### The Conjunction Fallacy (the "Linda Problem")

This is a well-known experiment in cognitive psychology (Tversky & Kahneman, 1983). It exploits the **conjunction fallacy**: people rate a conjunction $P(A \cap B)$ as more likely than one of its components $P(A)$ alone, which is mathematically impossible:

$$P(A \cap B) \leq P(A)$$

LLMs trained on human text inherit this same bias.

Key question to watch: even when the model reasons step by step, does it still pick the logically impossible answer because the description *feels* more consistent with it?

In [13]:
linda_problem = """Linda is 31 years old, single, outspoken, and very bright. She majored in philosophy.
As a student, she was deeply concerned with issues of discrimination and social justice,
and also participated in anti-nuclear demonstrations.

Which is more probable?
A) Linda is a bank teller.
B) Linda is a bank teller and is active in the feminist movement."""

print('Question:')
print(linda_problem)
print('\nAnswer (no CoT):')
print(call_llm(linda_problem))

Question:
Linda is 31 years old, single, outspoken, and very bright. She majored in philosophy.
As a student, she was deeply concerned with issues of discrimination and social justice,
and also participated in anti-nuclear demonstrations.

Which is more probable?
A) Linda is a bank teller.
B) Linda is a bank teller and is active in the feminist movement.

Answer (no CoT):
Linda is described as a 31-year-old, single individual, and a philosophy major, who is deeply concerned with issues of discrimination and social justice, and has participated in anti-nuclear demonstrations. The question asks which of the two options (A) or (B) is more probable based on these traits.

---

**Option A** states that Linda is a bank teller. While this is a plausible profession for someone of her age, it is not directly tied to her personal characteristics such as being a philosophy major or deeply involved in social justice. It does not clearly align with her stated interests.

**Option B** states that Li

In [14]:
print('Answer (with CoT):')
print(call_llm(add_cot(linda_problem)))

Answer (with CoT):
To determine which is more probable, we must analyze the information given about Linda and assess which option logically aligns with her attributes.

---

### Key Information:

- **Linda is 31 years old, single, and has a philosophy major.**
- She is **outspoken**, and **concerned with issues of discrimination and social justice**.
- She **participated in anti-nuclear demonstrations**.

---

### Option Analysis:

**A) Linda is a bank teller.**  
This option suggests that Linda has a specific job. However, there is **no direct evidence** in the information provided that supports this conclusion. While being a bank teller is a job she might have, the question doesn’t provide enough information to confirm her occupation beyond being a teller.

**B) Linda is a bank teller and is active in the feminist movement.**  
This option aligns with the information given. Being a **bank teller** and being **active in feminist issues** (anti-nuclear demonstrations, social justice) i

> **Observe:** Even with step-by-step reasoning, the model often still picks B. The description *feels* like a feminist activist, and the model's reasoning is retrofitted to justify that pre-formed impression. This is the hallmark of motivated reasoning.

**Prompt chaining** breaks the task into isolated sub-problems, preventing the model from jumping to the emotionally satisfying answer. The key insight: by asking the model to commit to the relevant rule *in isolation* (before it even sees the Linda description), we anchor its reasoning.

The chain:
1. **Step A**: ask the model to explain the probability rule for conjunctions, in a neutral context with no mention of Linda.
2. **Step B**: inject the rule the model just stated and ask it to apply *only* that rule to the Linda problem.

In [18]:
# Step A: establish the rule without any context that might trigger the fallacy
rule_prompt = (
    'In probability theory, if A and B are two events, '
    'how does P(A and B) compare to P(A) alone? '
    'Explain in one or two sentences, with the inequality, which of the two probabilities is larger.'
)
print('Step A - establishing the probability rule')
rule = call_llm(rule_prompt, temperature=0)
print(rule)

print()

# Step B: inject the established rule as a given and apply it to Linda
application_prompt = (
    f'You previously stated this probability rule:\n"{rule}"\n\n'
    'Using ONLY this mathematical rule — and ignoring how well the description '
    'matches each option — decide which option has a higher probability:\n\n'
    + linda_problem
    + '\n\nApply the rule directly. State your answer and a one-sentence justification.'
)
print('Step B - applying the rule to the Linda problem:')
print(call_llm(application_prompt, temperature=0))

Step A - establishing the probability rule


ReadTimeout: HTTPConnectionPool(host='localhost', port=11434): Read timed out. (read timeout=60)

> **Observe:** By first making the model commit to the probability rule in isolation, and then injecting that rule as a premise, we significantly reduce the chance that the model reasons backwards from the emotionally appealing answer.
>
> Prompt chaining turns the model's own output into a reasoning constraint. This is especially valuable when the task involves a principle that the model "knows" intellectually but fails to apply when distracted by irrelevant but vivid context.

## Step 5: Tree of Thoughts (ToT)

Chain-of-Thought follows a single reasoning path. For problems where the right approach is not obvious upfront, a single chain risks going down a dead end and not recovering.

**Tree of Thoughts** (Yao et al., 2023) extends CoT by:
1. Generating **multiple candidate reasoning branches** from the same starting point
2. **Evaluating** each branch (by the model or by a heuristic)
3. **Selecting** the most promising branch and continuing from there

This mimics deliberate planning. Although it has a computational performance trade-off: it implies `1 + B` LLM calls per reasoning level, where B is the branching factor.

---

We implement a simple single-level ToT: generate B independent reasoning starts, ask the model to evaluate them, then continue from the best one.

### Problem: Finding the Heavy Coin

A classic algorithmic puzzle with multiple valid strategies, although some far more efficient than others. The optimal solution (divide into groups of 3, not 2 or 4) is non-obvious without careful analysis.

In [ ]:
tot_problem = (
    'You have 9 coins that look identical. Exactly one coin is slightly heavier than the others. '
    'You have a balance scale and may use it exactly twice. '
    'Describe a strategy that guarantees identifying the heavy coin.'
)

# ── Step 1: Generate multiple reasoning branches ────────────────────────────────
branch_prompt = (
    f'Problem: {tot_problem}\n\n'
    'Generate 3 different possible initial approaches to solve this problem. '
    'Number them 1, 2, 3. Be concise — one or two sentences per approach.'
)
print('Problem:', tot_problem)
print()
print('── Step 1: Generate reasoning branches ──')
branches = call_llm(branch_prompt, temperature=0.8)
print(branches)

print()

# ── Step 2: Evaluate and select the most promising branch ──────────────────────
eval_prompt = (
    f'Problem: {tot_problem}\n\n'
    f'Candidate approaches:\n{branches}\n\n'
    'Which numbered approach is most likely to lead to a correct solution that uses '
    'the balance scale exactly twice? State the number and explain why in one or two sentences.'
)
print('── Step 2: Evaluate branches ──')
selection = call_llm(eval_prompt, temperature=0)
print(selection)

print()

# ── Step 3: Continue with the selected branch to a full solution ────────────────
solve_prompt = (
    f'Problem: {tot_problem}\n\n'
    f'Candidate approaches:\n{branches}\n\n'
    f'Selected approach: {selection}\n\n'
    'Now work out the complete step-by-step solution using the selected approach. '
    'Be explicit about what you place on each side of the scale at each weighing, '
    'and what each outcome tells you.'
)
print('── Step 3: Full solution from the selected branch ──')
print(call_llm(solve_prompt, temperature=0))

> **Observe:**
> - ToT gives visibility into the model's *strategy space*, not just one chosen path. This is more transparent than a single chain that commits immediately to an approach.
> - Even if the final solution is imperfect, the evaluation step is itself informative: it shows whether the model can correctly judge the quality of reasoning strategies.
> - For very small models like `qwen3:0.6b`, the quality of each step may vary. The value of the pattern is clearer with larger models, where branch quality differences are more meaningful.
> - **Exercise:** Try increasing the number of branches from 3 to 5. Does the model find a better solution? Does the evaluation step become more useful?

## Step 6: Reasoning Failures on Modern Models

`qwen3:0.6b` is a very small model, so reasoning failures are easy to trigger. But reasoning failures are not exclusive to small models — certain carefully constructed problems expose failures even in frontier models like GPT-4, Gemini, and Claude Sonnet.

The key property of such problems: the problem **looks** like a well-known pattern the model has memorised, but contains a subtle modification that invalidates the memorised reasoning. The model pattern-matches to the familiar form and applies the wrong logic.

---

### The Unexpected Hanging Paradox — Modified

The classic version of this paradox has been widely discussed online and in philosophy texts, so LLMs have seen many explanations of it. The **modified version** below introduces one key change: the prisoner is told the execution day **on Monday morning** — a fixed, known date — rather than being surprised on the day itself.

This single change invalidates the prisoner's backwards-induction argument, but most frontier models miss it and apply the reasoning from the classic version instead.

**Run the cells below with your local `qwen3:0.6b`, then — as an exercise — paste the same prompts into Gemini or Claude Sonnet and compare.**

In [ ]:
hanging_paradox = (
    'Imagine a judge tells a prisoner that he will be hanged at noon on one weekday '
    'in the following week but that the execution will be a surprise to the prisoner. '
    'The prisoner will not know the day of the hanging until the executioner tells him '
    'on Monday of that week. '
    'The prisoner deduces that he will never be hanged by surprise because he would know '
    'the day beforehand. The prisoner is executed on a Friday. '
    'Was the execution a surprise to the prisoner?'
)

print('Question (no CoT):')
print(hanging_paradox)
print()
print('Answer:')
print(call_llm(hanging_paradox))

The model likely said the execution *was* a surprise (or reproduced reasoning from the classic version of the paradox). Now try the same question with CoT guidance that explicitly directs attention to the one detail that matters: **when** the prisoner is told.

In [ ]:
hanging_cot = (
    hanging_paradox
    + '\n\n'
    'Analyze the problem step by step. Do not use previously acquired knowledge or '
    'assumptions about this paradox to solve it. First, check the date and time when '
    'the prisoner will be told the day of the hanging, and establish whether this is '
    'a fixed date or varies. Then reason from that fact alone.'
)

print('Question (with CoT):')
print(hanging_cot)
print()
print('Answer:')
print(call_llm(hanging_cot))

## Summary and Exercises

| Technique | When to use | Cost |
|---|---|---|
| **Zero-shot CoT** | Any multi-step reasoning task; near-free | 1 LLM call |
| **Prompt chaining** | Motivated reasoning; tasks with separable sub-problems | N LLM calls (one per step) |
| **Tree of Thoughts** | Complex planning; problems with non-obvious strategies | 1 + B LLM calls per level |

---

### Exercises

1. **Threshold tuning (CoT):** Modify `add_cot` to pass few-shot examples of correct reasoning instead of just an instruction. Does *few-shot CoT* outperform *zero-shot CoT* on the bat-and-ball problem?

2. **Chaining failure modes:** The Linda prompt-chaining approach works because the rule can be established neutrally. Can you think of a problem where prompt chaining would *also* fail? (Hint: what if the model's answer to Step A is itself biased?)

3. **ToT branching factor:** Re-run the coin problem with `temperature=0` for branch generation. Do you still get diverse branches? What does this tell you about the role of temperature in ToT?

4. **Modern model test:** Paste the hanging paradox (without CoT) into Gemini or Claude Sonnet. Compare the answer to the CoT version. Does the modification (`told on Monday`) change the difficulty level compared to `qwen3:0.6b`?

5. **Bring your own problem:** Find or craft a problem where the correct answer requires ignoring a vivid but irrelevant detail. Test it with and without CoT, and decide whether prompt chaining or ToT is the more appropriate transparency technique.